In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura a renderização gráfica inline no Jupyter notebook
%matplotlib inline

# O pré-treinamento em estado de repouso se transfere para tempo de reação?

Carregue três participantes do R5 mini e a execução (*run*) 1 da detecção de mudança de contraste.
Preveja o tempo de estímulo à resposta em segundos a partir dos dois segundos precedentes
de EEG. A divisão retém um participante completo (*held-out*); as amostras selecionadas terminam
no início do estímulo (*stimulus onset*). Pré-treine uma EEGNet em pistas observadas de olhos abertos/fechados
dos dois participantes de treino e, em seguida, adapte seu codificador para regressão do tempo de reação.
Compare com uma rede de formato idêntico treinada do zero (*from scratch*). Nenhuma gravação do participante
de teste entra em qualquer estágio de treinamento. Orçamentos fixos de duas épocas ilustram as operações;
isso não constitui evidência de ganhos gerais de transferência.


## Antes de começar

Use um ambiente com EEGDash instalado juntamente com Braindecode, MNE, NumPy,
scikit-learn e Matplotlib; este script é executado em CPU. Mantenha um ``EEGDASH_CACHE_DIR``
persistente: as três gravações da execução 1 de mudança de contraste são baixadas por completo
no primeiro acesso, mesmo que cada exemplo use janelas curtas. A versão com transferência
também precisa de duas gravações de estado de repouso. Ambas as tarefas usam os derivados filtrados
do desafio a 100 Hz, 0.5–50 Hz.

O tempo de reação é uma latência contínua observada, não uma categoria rápido/lento. A
janela termina na âncora do estímulo. Isso exclui amostras pós-estímulo do intervalo selecionado,
mas não estabelece um pipeline causal em tempo real (*online*): a versão de origem já foi
filtrada e seu pré-processamento deve ser auditado separadamente antes de se afirmar predição em tempo real.



In [ ]:
# Importa utilitários do sistema operacional, caminhos e clonagem profunda de objetos
import os
from pathlib import Path
import copy

# Importa bibliotecas para plotagem e manipulação de tensores e matrizes numéricas
import matplotlib.pyplot as plt
import numpy as np
import torch

# Importa funções de janelamento, arquitetura EEGNet e wrappers de treinamento da Braindecode
from braindecode.preprocessing import create_windows_from_events
from braindecode.models import EEGNet
from braindecode import EEGClassifier, EEGRegressor
# Importa métrica de erro absoluto médio do scikit-learn
from sklearn.metrics import mean_absolute_error

# Importa classe do dataset do desafio e funções auxiliares de janelamento do HBN
from eegdash import EEGChallengeDataset
from eegdash.hbn.windows import (
    annotate_trials_with_target,
    add_aux_anchors,
    add_extras_columns,
)

## Carregar os participantes nomeados e eventos observados

O filtro de execução explícito impede que uma consulta por sujeito carregue todas as três
execuções de mudança de contraste. Os dois primeiros IDs de sujeitos treinarão o modelo; o terceiro
é reservado para avaliação. A asserção de cobertura de sujeitos detecta uma gravação ausente
em vez de alterar silenciosamente esse planejamento.

``annotate_trials_with_target`` lê o arquivo auxiliar de eventos da gravação e pareia ensaios
de contraste com os tempos reais de estímulo e resposta. Ensaios sem os eventos necessários
não fornecem uma latência observada. ``add_aux_anchors`` posiciona anotações nesses tempos
reais de estímulo; não cria rótulos de resposta. Inspecione os nomes das anotações impressos
e a taxa de 100 Hz antes do janelamento.



In [ ]:
# Lista de 3 sujeitos: os dois primeiros para treino e o terceiro retido para teste
subjects = ["NDARDC843HHM", "NDAREC480KFA", "NDARAP785CTE"]
# Configura diretório de cache persistente
cache = Path(os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")).expanduser()
# Carrega gravações da tarefa de mudança de contraste na execução 1 para os sujeitos escolhidos
dataset = EEGChallengeDataset(
    release="R5",
    mini=True,
    task="contrastChangeDetection",
    subject=subjects,
    run="1",
    cache_dir=cache,
)
# Exibe metadados descritivos das gravações carregadas
print(dataset.description.to_string(index=False))
# Assegura que todos os três sujeitos requisitados foram encontrados
assert set(dataset.description.subject) == set(subjects)
# Itera pelas gravações para selecionar canais EEG, validar taxa de amostragem e anotar eventos
for recording in dataset.datasets:
    raw = recording.raw
    # Filtra estritamente os canais de eletroencefalografia
    raw.pick("eeg")
    # Imprime informações de sujeito, canais, taxa de amostragem e descrições de anotações
    print(
        recording.description.subject,
        raw.ch_names,
        raw.info["sfreq"],
        np.unique(raw.annotations.description),
    )
    # Garante que a taxa de amostragem seja exatamente 100 Hz
    assert raw.info["sfreq"] == 100
    # Calcula o tempo de reação a partir do estímulo e anota os ensaios
    annotate_trials_with_target(raw, target_field="rt_from_stimulus")
    # Insere âncoras auxiliares nos instantes exatos de apresentação dos estímulos
    add_aux_anchors(raw)

## Extrair um preditor pré-estímulo de dois segundos

A 100 Hz, os deslocamentos ``-200`` e ``0`` selecionam o intervalo imediatamente
anterior ao estímulo. O tamanho e o passo de 200 amostras produzem uma janela para cada
âncora utilizável. ``mapping={"stimulus_anchor": 0}`` informa ao gerador de janelas quais âncoras
usar; esse zero é um código de seleção, não o alvo da regressão.

``add_extras_columns`` transfere o ``rt_from_stimulus`` medido para os metadados da janela.
Use essa coluna para ``y``, em segundos. Um array finito com formato ``(trials, canais de EEG, 200)``
fornece preditores em volts. Asserções de latência positiva e finita expõem pareamentos de eventos
malformados; elas não exigem um erro de predição específico ou resultado favorável.



In [ ]:
# Cria janelas de 2 segundos (200 amostras a 100 Hz) imediatamente anteriores à âncora do estímulo (-200 a 0)
windows = create_windows_from_events(
    dataset,
    mapping={"stimulus_anchor": 0},
    trial_start_offset_samples=-200,
    trial_stop_offset_samples=0,
    window_size_samples=200,
    window_stride_samples=200,
    preload=True,
)
# Adiciona colunas extras de metadados às janelas, incluindo o tempo de reação
windows = add_extras_columns(
    windows,
    dataset,
    desc="stimulus_anchor",
    keys=("target", "rt_from_stimulus", "stimulus_onset"),
)
# Obtém metadados das janelas com índice reiniciado
metadata = windows.get_metadata().reset_index(drop=True)
# Empilha os tensores de sinal de cada janela em uma matriz 3D (n_ensaios, n_canais, n_amostras)
X = np.stack([windows[i][0] for i in range(len(windows))])
# Extrai os tempos de reação em segundos como array de ponto flutuante
y = metadata.rt_from_stimulus.to_numpy(dtype=float)
# Valida que os sinais e alvos são finitos e que todos os tempos de reação são estritamente positivos
assert np.isfinite(X).all() and np.isfinite(y).all() and (y > 0).all()
# Cria máscara booleana para treino (dois primeiros sujeitos)
train = metadata.subject.isin(subjects[:2]).to_numpy()
# Cria máscara booleana para teste (terceiro sujeito retido)
test = metadata.subject.eq(subjects[2]).to_numpy()
# Assegura presença de dados em ambos os conjuntos e separação disjunta estrita de sujeitos
assert train.any() and test.any()
assert set(metadata.subject[train]).isdisjoint(metadata.subject[test])
# Conversão fixa para microvolts: multiplica por 1e6 sem estimar transformações em dados de teste
X = X.astype("float32") * 1e6

## Pré-treinar em uma tarefa auxiliar observada

Apenas os dois participantes de treino fornecem EEG em repouso. Verificar seus
IDs contra o participante-alvo retido fecha um vazamento comum em aprendizado por transferência:
excluir uma pessoa do ajuste fino (*fine-tuning*) é insuficiente se sua gravação já tiver
sido utilizada durante o pré-treinamento.

Os rótulos de origem codificam instruções para abrir (0) ou fechar (1) os olhos. O
deslocamento de um segundo afasta a janela de dois segundos do início da instrução;
esses rótulos refletem o protocolo, não uma medição independente da posição dos olhos.
Este é um pré-treinamento supervisionado em tarefa auxiliar, não autosupervisão.
Tanto o conjunto de origem quanto o de destino usam a mesma ordem de canais e comprimento de
200 amostras. Multiplicar volts por ``1e6`` fornece microvolts sem ajustar nenhuma
estatística nos participantes retidos.



In [ ]:
# Carrega o dataset de estado de repouso apenas para os dois sujeitos de treino
source = EEGChallengeDataset(
    release="R5", mini=True, task="RestingState", subject=subjects[:2], cache_dir=cache
)
# Garante que nenhum sujeito presente no teste esteja na fonte de pré-treino
assert set(source.description.subject).isdisjoint(metadata.subject[test])
# Valida que os canais de EEG são idênticos aos da tarefa de tempo de reação
for recording in source.datasets:
    recording.raw.pick("eeg")
    assert recording.raw.ch_names == dataset.datasets[0].raw.ch_names
# Cria janelas estáveis de 2 segundos (deslocamento de 100 a 300 amostras após a instrução)
source_windows = create_windows_from_events(
    source,
    mapping={"instructed_toOpenEyes": 0, "instructed_toCloseEyes": 1},
    trial_start_offset_samples=100,
    trial_stop_offset_samples=300,
    window_size_samples=200,
    window_stride_samples=200,
    preload=True,
)
# Empilha os tensores de sinal de repouso e converte para microvolts (float32)
Xs = (
    np.stack([source_windows[i][0] for i in range(len(source_windows))]).astype(
        "float32"
    )
    * 1e6
)
# Extrai os rótulos de olhos abertos (0) e fechados (1)
ys = np.asarray([source_windows[i][1] for i in range(len(source_windows))])
# Assegura finitude dos sinais e presença das duas classes
assert np.isfinite(Xs).all() and set(ys) == {0, 1}
# Imprime formato das janelas e contagens de classes na tarefa de repouso
print(
    "Resting-state windows:",
    Xs.shape,
    "real eye-state counts:",
    np.unique(ys, return_counts=True),
)
# Define semente pseudoaleatória e número de threads no PyTorch para determinismo e controle
torch.manual_seed(71)
torch.set_num_threads(2)

## Treinar o codificador de origem e substituir sua cabeça (*head*)

A EEGNet aprende filtros temporais e espaciais diretamente das janelas.
A cabeça de origem retorna dois logits para entropia cruzada; a cabeça posterior
retorna um número real para regressão com erro quadrático. O passo do Adam de ``1e-3``,
duas épocas e lotes de 16 exemplos (com um lote final menor) são parâmetros didáticos fixos,
não hiperparâmetros selecionados para esta coorte. ``EEGClassifier`` e
``EEGRegressor`` gerenciam lotes, gradientes e modo de avaliação;
``train_split=None`` evita uma divisão adicional de validação no nível de janelas.
Alvos de regressão mantêm o formato ``(trials, 1)`` para corresponder à cabeça de saída única.

A padronização do alvo usa apenas as latências de treino. As predições são posteriormente
multiplicadas pelo desvio padrão de treino e somadas à média de treino para retornar à escala em segundos.
Copiar todo o estado exceto a ``final_layer`` transfere o codificador enquanto mantém a nova
cabeça de uma saída. A asserção de chave ausente garante que o carregamento relaxado não descartou
parâmetros não relacionados silenciosamente. Ambas as condições posteriores começam com a mesma cabeça
inicializada aleatoriamente.



In [ ]:
# Cria arquitetura EEGNet configurada para classificação binária (olhos abertos vs. fechados)
encoder = EEGNet(n_chans=X.shape[1], n_outputs=2, n_times=200, sfreq=100)
# Configura o treinador do classificador com CrossEntropyLoss e otimizador Adam
source_trainer = EEGClassifier(
    encoder,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    lr=1e-3,
    batch_size=16,
    max_epochs=2,
    train_split=None,
    iterator_train__shuffle=False,
    classes=[0, 1],
    device="cpu",
)
# Ajusta o modelo na tarefa auxiliar de repouso
source_trainer.fit(Xs, ys)
# Recupera o módulo PyTorch treinado
encoder = source_trainer.module_
# Calcula a média e o desvio padrão dos tempos de reação estritamente no conjunto de treino
mean, scale = y[train].mean(), y[train].std()
# Garante desvio padrão estritamente positivo para padronização
assert scale > 0
# Dicionário para armazenar resultados de ambos os regimes
results = {}
# Cria o modelo base de regressão (saída única) para inicialização idêntica
initial = EEGNet(n_chans=X.shape[1], n_outputs=1, n_times=200, sfreq=100)
# Compara os regimes: do zero ('from scratch') versus transferência de repouso ('resting-state transfer')
for regime in ["from scratch", "resting-state transfer"]:
    # Clona a arquitetura inicial garantindo que ambos iniciam com a mesma cabeça
    model = copy.deepcopy(initial)
    if regime == "resting-state transfer":
        # Filtra os pesos do codificador de repouso excluindo a camada final de classificação
        state = {
            k: v
            for k, v in encoder.state_dict().items()
            if not k.startswith("final_layer")
        }
        # Carrega os pesos transferidos no codificador
        missing, unexpected = model.load_state_dict(state, strict=False)
        # Valida que apenas as camadas da final_layer estavam ausentes no state_dict carregado
        assert not unexpected and all(k.startswith("final_layer") for k in missing)
    # Configura o regressor com MSELoss e taxa de aprendizado 1e-3
    regressor = EEGRegressor(
        model,
        criterion=torch.nn.MSELoss,
        optimizer=torch.optim.Adam,
        lr=1e-3,
        batch_size=16,
        max_epochs=2,
        train_split=None,
        iterator_train__shuffle=False,
        device="cpu",
    )
    # Padroniza os valores de y do treino e ajusta dimensões para (n_amostras, 1)
    standardized_y = ((y[train] - mean) / scale).astype("float32")[:, None]
    # Treina o regressor nos ensaios de treino
    regressor.fit(X[train], standardized_y)
    # Realiza predições no participante de teste e reverte a padronização para a escala original em segundos
    predicted = regressor.predict(X[test]).reshape(-1) * scale + mean
    # Calcula o erro absoluto médio (MAE) no participante retido
    results[regime] = mean_absolute_error(y[test], predicted)
# Exibe os valores de MAE calculados
print("Held-out reaction-time MAE (s):", results)
# Plota gráfico de barras comparativo entre os dois regimes
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(results), list(results.values()))
ax.set(ylabel="Held-out participant trial MAE (s)")
plt.show()

## Interpretar a transferência sem selecionar no participante de teste

Cada barra representa o erro absoluto médio sobre os ensaios utilizáveis da mesma pessoa
retida; menor é melhor e um erro de 0.1 significa 100 ms em média. A condição de transferência
recebe treinamento extra na tarefa de origem, portanto a comparação não é equalizada
pelo total de passos de otimização. Um sujeito e uma inicialização não conseguem demonstrar um
benefício geral confiável de transferência. O gráfico pode favorecer qualquer uma das condições.

Para estender o experimento, reserve participantes adicionais de validação antes de
qualquer estágio de treinamento. Selecione o orçamento de épocas e a taxa de aprendizado ali e,
em seguida, repita toda a comparação sobre participantes de teste e sementes intocados. Também
compare com um preditor baseado na média de latência do treino e avalie exclusões por respostas
ausentes. Não ajuste tarefas de origem após olhar para essas barras de teste.

Exemplos práticos relacionados: [cross-dataset transfer](https://braindecode.org/dev/auto_examples/advanced_training/plot_transfer_learning.html)
e [relative-positioning pretraining](https://braindecode.org/dev/auto_examples/advanced_training/plot_relative_positioning.html).

